# Variant fixation in 1000 Genomes + HGDP

**Reviewer #3** (minor) and **Reviewer #4** (comment 1) asked whether the assayed human-derived variants are also fixed in more diverse cohorts (1KG/HGDP), and what fraction of the genome was assessable. **Response:** re-checked allele frequencies in gnomAD v4 and a combined 1KG+HGDP set (gnomAD v3.1); 99.96% of variants have frequency >0.999.

In [ ]:
import pandas as pd
from Bio import SeqIO
import sys
import os
from ast import literal_eval
import numpy as np
import pybedtools
from pybedtools import BedTool
from matplotlib.colors import LinearSegmentedColormap
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy import stats
from scipy.stats import gaussian_kde
from scipy.stats import pearsonr
from scipy.stats import spearmanr
import itertools
import re
import ast
import math

# --- Shared helpers: const.py lives in analyses/common ---
NB_DIR = os.getcwd()
sys.path.append(os.path.abspath(os.path.join(NB_DIR, '..', 'common')))
import const
from const import pos_active_ctrl_color, neg_active_ctrl_color, highlight_color, custom_cmap
from const import set_equal_plot_limits, plot_color_pallete, custom_cmap_bolder, FONT_SIZE_small
const.set_plot_style()

# --- Output directory (EDIT): where plots/tables are written ---
output_path = '.'

# --- External liftOver tool (EDIT): used to convert genome coordinates ---
LIFTOVER_DIR = '/home/labs/davidgo/Collaboration/Lab_Tools/liftOver/no_GUI'
os.chdir(LIFTOVER_DIR)
from liftOver import initializer as initializer
from liftOver import liftOver_df as liftOver
os.chdir(NB_DIR)


# Check the frequencies of hMPRA variants in 1K and HGDP

## Create a BED file of all SNPs in the library

In [ ]:
# Load the library design and create the BED file
library_design = pd.read_excel("/home/labs/davidgo/Collaboration/humanMPRA/paper/extended_datasets/combined_library_design.xlsx")

In [ ]:
from pathlib import Path
import re

# Parse genomic coordinates from the library_design "Variants (genomic)" column.
# Keep these coordinates 1-based for liftover input.
variants_col = "Variants (genomic)" if "Variants (genomic)" in library_design.columns else "variants_genomic"
coord_pattern = re.compile(r"(chr[\w]+):(\d+)(?:-(\d+))?")

records = []
unparsed_rows = []

for idx, row in library_design.iterrows():
    raw = row.get(variants_col)
    if pd.isna(raw):
        continue

    matches = coord_pattern.findall(str(raw))
    if not matches:
        unparsed_rows.append(idx)
        continue

    oligo_id = row.get("ID") if "ID" in library_design.columns else idx
    for chrom, start, end in matches:
        start_i = int(start)
        end_i = int(end) if end else start_i

        # Keep 1-based coordinates for liftover tool input.
        records.append((chrom, start_i, end_i, oligo_id, str(raw)))

variants_bed = pd.DataFrame(records, columns=["chrom", "start", "end", "id", "variant"])
variants_bed = variants_bed.sort_values(["chrom", "start", "end", "id"]).drop_duplicates()

# 3-column liftover input table (1-based coordinates; BED conversion happens after liftover).
positions_bed = variants_bed[["chrom", "start", "end"]].drop_duplicates().sort_values(["chrom", "start", "end"])

base_output = Path(output_path) if "output_path" in globals() else Path.cwd()
base_output.mkdir(parents=True, exist_ok=True)

input_coords_path = base_output / "library_design_variants_genomic_positions_for_liftover_1based.tsv"
positions_bed.to_csv(input_coords_path, sep="\t", header=False, index=False)

# Optional metadata table keeps oligo ID and original variant string (still 1-based).
metadata_bed_path = base_output / "library_design_variants_genomic_positions_for_liftover_1based.with_metadata.tsv"
variants_bed.to_csv(metadata_bed_path, sep="\t", header=False, index=False)

print(f"Wrote {len(positions_bed):,} unique 1-based coordinates to: {input_coords_path}")
print(f"Wrote {len(variants_bed):,} 1-based rows with metadata to: {metadata_bed_path}")
print(f"Rows with no parseable coordinates: {len(unparsed_rows):,}")

In [ ]:
lifted_coords = liftOver(positions_bed,chr_col = 'chrom',start_col= 'start', end_col= 'end',source_assembly= 'hg19',target_assembly='hg38')


In [ ]:
# Convert both coordinate sets to proper BED convention (0-based start, half-open end) after liftover.
# Original hg19 coordinates (currently 1-based)
positions_bed.rename(columns={'chrom': 'hg19_chr', 'start': 'hg19_start', 'end': 'hg19_end', 'lift_name': 'coord'}, inplace=True)
positions_bed['hg19_start'] = positions_bed['hg19_start'].astype(int) - 1
positions_bed['hg19_end'] = positions_bed['hg19_end'].astype(int)

# Lifted hg38 coordinates (currently 1-based from liftover output)
lifted_coords.rename(columns={0: 'hg38_chr', 1: 'hg38_start', 2: 'hg38_end', 3: 'coord'}, inplace=True)
lifted_coords['hg38_start'] = lifted_coords['hg38_start'].astype(int)
lifted_coords['hg38_end'] = lifted_coords['hg38_end'].astype(int)

In [ ]:

# Merge on the coord column, keeping both coordinate sets
positions_bed_lift = positions_bed.merge(lifted_coords, left_on='coord', right_on='coord', how='right')

# Keep relevant columns: both coordinate sets, ID, and class
cols_to_keep = ['hg19_chr', 'hg19_start', 'hg19_end','hg38_chr', 'hg38_start', 'hg38_end']
positions_bed_lift = positions_bed_lift[cols_to_keep]

# Rename ID and class columns
positions_bed_lift.rename(columns={'3_x': 'ID'}, inplace=True)


In [ ]:
positions_bed_lift.to_csv('/home/labs/davidgo/Collaboration/humanMPRA/oligo_fasta/bed_files/variant_positions_bed_lift_w_header.tsv', sep="\t", header=True, index=False)

## Read the variant frequency tables 
(ie, the files generated by extract_1kg_frequency.py)

In [ ]:
positions_bed_lift = pd.read_csv('/home/labs/davidgo/Collaboration/humanMPRA/oligo_fasta/bed_files/variant_positions_bed_lift_w_header.tsv', sep="\t")

### gnomad v4 (which includes both HGDP and 1KG)

In [ ]:
def prepare_frequency_table(
    path,
    sep="\t",
    filter_col="filter",
    numeric_cols=("AC", "AN", "AF_global"),
    collapse_alt_col="alt",
    collapse_ref_col="ref",
    collapse_variant_id_cols=("variant_id",),
    aggregate_an="median",
    af_col="AF_global",
):
    """Load, QC-filter, aggregate duplicate variants, print summary stats, and return table."""
    table = pd.read_csv(path, sep=sep, low_memory=False)
    orig_cols = table.columns.tolist()

    print(f"Number rows before QC filtering: {len(table):,}")
    table = table[table[filter_col].isna() | (table[filter_col] == "PASS")].copy()
    print(f"Number rows after QC filtering: {len(table):,}")

    # Collapse REF/ALT and present variant-id columns.
    variant_id_cols_present = [c for c in collapse_variant_id_cols if c in table.columns]
    collapse_cols = [c for c in [collapse_ref_col, collapse_alt_col] if c in table.columns] + variant_id_cols_present

    # Group by all non-aggregated columns, excluding collapsed columns.
    group_cols = [
        c for c in orig_cols
        if c not in (set(numeric_cols) | set(collapse_cols))
    ]

    for c in numeric_cols:
        if c in table.columns:
            table[c] = pd.to_numeric(table[c], errors="coerce").fillna(0)

    def _collapse_unique(s):
        return ",".join(pd.unique(s.dropna().astype(str)))

    agg_spec = {
        "AC": ("AC", "sum"),
        "AN": ("AN", aggregate_an),
        "AF_global": ("AF_global", "sum"),
    }
    for c in collapse_cols:
        agg_spec[c] = (c, _collapse_unique)

    # Check for other potentially collapsible columns (report only, do not apply).
    key_cols = table.columns[:3].tolist()
    dup_mask = table.duplicated(subset=key_cols, keep=False)
    additional_collapse_candidates = []
    if dup_mask.any():
        dup_df = table.loc[dup_mask].copy()
        protected = set(numeric_cols) | set(collapse_cols) | {filter_col} | set(key_cols)
        for c in dup_df.columns:
            if c in protected:
                continue
            nunique_by_key = dup_df.groupby(key_cols, dropna=False)[c].nunique(dropna=True)
            if (nunique_by_key > 1).any():
                additional_collapse_candidates.append(c)

    table = table.groupby(group_cols, dropna=False, as_index=False).agg(**agg_spec)

    af_numeric = pd.to_numeric(table[af_col], errors="coerce")
    avg_af_missing_as_zero = float(af_numeric.fillna(0.0).mean())

    print(f"Collapsed columns used: {collapse_cols}")
    if additional_collapse_candidates:
        print("Potential additional collapsible columns (check only):")
        print(additional_collapse_candidates)
    else:
        print("Potential additional collapsible columns (check only): none detected")

    print(f"Rows in table after collapsing based on variant position: {len(table):,}")
    print(f"Unique variants after QC filtering (first 3 cols): {table.iloc[:, :3].drop_duplicates().shape[0]:,}")
    print(f"Average allele frequency (missing as 0): {avg_af_missing_as_zero:.10f}")
    af_for_pct = pd.to_numeric(table[af_col], errors="coerce").fillna(0.0)

    # Use float64 to maintain precision during the mean calculation
    pct_gt_1 = (af_for_pct > 0.01).astype('float64').mean() * 100
    pct_gt_01 = (af_for_pct > 0.001).astype('float64').mean() * 100
    
    print(f"% of variants with freq >1%: {pct_gt_1:.10f}%")
    print(f"% of variants with freq >0.1%: {pct_gt_01:.10f}%")

    return table


In [ ]:
# Apply once on gnomAD v4 table using the single helper function
path_gnomad_4 = "/home/labs/davidgo/Collaboration/humanMPRA/additional/variant_frequency_gnomad_v4/positions_bed_lift_gnomad_frequency.tsv"

table = prepare_frequency_table(path_gnomad_4)
freq_1kg_table = table  # Backward-compatible alias for downstream cells

In [ ]:
100-0.0397991673
100-0.0054026019

In [ ]:
zero_af_count = (freq_1kg_table["AF_global"] == 0).sum()
total_rows = len(freq_1kg_table)
zero_af_percent = (zero_af_count / total_rows) * 100 if total_rows > 0 else np.nan

print(f"Rows with AF_global == 0: {zero_af_count:,}")
print(f"Percent with AF_global == 0: {zero_af_percent:.4f}%")

In [ ]:
sum(freq_1kg_table['AF_global']==0)

### check for duplicated rows after collapsing

In [ ]:
def find_duplicated_rows_by_first_cols(df, n_cols=3, sort_cols=("hg19_chr", "hg19_start"), verbose=True):
    subset_cols = df.columns[:n_cols]
    dup_mask = df.duplicated(subset=subset_cols, keep=False)

    duplicated_rows = df.loc[dup_mask].copy()
    if all(col in duplicated_rows.columns for col in sort_cols):
        duplicated_rows = duplicated_rows.sort_values(list(sort_cols))
    duplicated_rows = duplicated_rows.reset_index(drop=True)

    n_dups = len(duplicated_rows)

    if verbose:
        print(f"Columns used: {list(subset_cols)}")
        print(f"Duplicated rows (based on cols 1-{n_cols}): {n_dups:,}")

    return duplicated_rows, dup_mask, n_dups, subset_cols


duplicated_rows, dup_mask, n_dups, subset_cols = find_duplicated_rows_by_first_cols(freq_1kg_table)

### Histogram of AFs

In [ ]:
def plot_af_global_histogram(df, af_col="AF_global", bins_count=60):
    """Plot AF histogram on a log10 x-axis and return plotting intermediates."""
    af = pd.to_numeric(df[af_col], errors="coerce")
    af_pos = af[(af > 0) & af.notna()]

    if af_pos.empty:
        raise ValueError(f"No positive values found in {af_col}.")

    fig, ax = plt.subplots(figsize=(8, 5))
    bins = np.logspace(np.log10(af_pos.min()), np.log10(af_pos.max()), bins_count)
    ax.hist(af_pos, bins=bins, color=plot_color_pallete["barcode"], edgecolor="white", alpha=0.8)

    ax.set_xscale("log", base=10)
    ax.set_xlabel("AF_global (log10 scale)", fontsize=FONT_SIZE_small)
    ax.set_ylabel("Count", fontsize=FONT_SIZE_small)
    ax.set_title("Histogram of AF_global (x-axis in log10)", fontsize=FONT_SIZE_small)

    plt.tight_layout()
    plt.show()

    return fig, ax, af, af_pos


fig, ax, af, af_pos = plot_af_global_histogram(freq_1kg_table)
print(f"Plotted {len(af_pos):,} variants with AF_global > 0")
print(f"Excluded {(af <= 0).sum():,} rows with AF_global <= 0")

In [ ]:
def build_validation_rows(library_design_df, positions_bed_lift_df, freq_table):
    """Build validation rows by mapping hMPRA variants to lifted coords and 1KG frequency rows."""
    variants_col = "Variants (genomic)" if "Variants (genomic)" in library_design_df.columns else "variants_genomic"

    # 1) Explode rows so each variant token is a separate row
    variants_expanded = library_design_df[[variants_col]].copy()
    variants_expanded = variants_expanded.dropna().reset_index().rename(columns={"index": "library_row_idx"})
    variants_expanded["variant_token"] = variants_expanded[variants_col].astype(str).str.split(";")
    variants_expanded = variants_expanded.explode("variant_token")
    variants_expanded["variant_token"] = variants_expanded["variant_token"].astype(str).str.strip()

    # Parse tokens like chr1:877023(T->G)
    token_pattern = re.compile(
        r"^(chr[\w]+):(\d+)(?:-(\d+))?\s*\(\s*([ACGTN])\s*->\s*([ACGTN])\s*\)$",
        flags=re.IGNORECASE,
    )

    parsed_rows = []
    for _, row in variants_expanded.iterrows():
        tok = row["variant_token"]
        m = token_pattern.match(tok)
        if not m:
            continue
        parsed_rows.append(
            (
                int(row["library_row_idx"]),
                m.group(1),
                int(m.group(2)),
                m.group(4).upper(),
                m.group(5).upper(),
                tok,
            )
        )

    library_alleles = pd.DataFrame(
        parsed_rows,
        columns=[
            "library_row_idx",
            "hg19_chr",
            "hg19_pos1",
            "library_left",
            "library_right",
            "variant_token",
        ],
    ).drop_duplicates()

    # 2) Build hg19->hg38 map from lifted coordinates
    lift_map = positions_bed_lift_df.copy()
    lift_map["hg19_chr"] = lift_map["hg19_chr"].astype(str)
    lift_map["hg38_chr"] = lift_map["hg38_chr"].astype(str)
    for c in ["hg19_start", "hg38_start", "hg38_end"]:
        lift_map[c] = pd.to_numeric(lift_map[c], errors="coerce")
    lift_map = lift_map.dropna(subset=["hg19_start", "hg38_start", "hg38_end"]).copy()
    lift_map["hg19_start"] = lift_map["hg19_start"].astype(int)
    lift_map["hg38_start"] = lift_map["hg38_start"].astype(int)
    lift_map["hg38_end"] = lift_map["hg38_end"].astype(int)
    lift_map["hg19_pos1"] = lift_map["hg19_start"] + 1

    alleles_lifted = library_alleles.merge(
        lift_map[["hg19_chr", "hg19_pos1", "hg38_chr", "hg38_start", "hg38_end"]].drop_duplicates(),
        on=["hg19_chr", "hg19_pos1"],
        how="left",
    )

    # 3) Found-in-1KG table, deduplicated at variant row level
    freq_found = freq_table.copy()
    freq_found = freq_found[freq_found["found_in_1kg"].fillna(False)].copy()
    freq_found["hg38_chr"] = freq_found["hg38_chr"].astype(str)
    for c in ["hg38_start", "hg38_end"]:
        freq_found[c] = pd.to_numeric(freq_found[c], errors="coerce")
    freq_found = freq_found.dropna(subset=["hg38_start", "hg38_end"]).copy()
    freq_found["hg38_start"] = freq_found["hg38_start"].astype(int)
    freq_found["hg38_end"] = freq_found["hg38_end"].astype(int)
    freq_found = freq_found.drop_duplicates(
        subset=["hg38_chr", "hg38_start", "hg38_end", "ref", "alt", "AF_global"]
    )

    # 4) Compare per exploded variant annotation
    ann_unique = alleles_lifted[[
        "hg38_chr", "hg38_start", "hg38_end",
        "library_left", "library_right", "variant_token",
    ]].dropna(subset=["hg38_chr", "hg38_start", "hg38_end"]).drop_duplicates()

    validation_rows = freq_found.merge(
        ann_unique,
        on=["hg38_chr", "hg38_start", "hg38_end"],
        how="inner",
    )

    validation_rows["AF_num"] = pd.to_numeric(validation_rows["AF_global"], errors="coerce")
    validation_rows["ref_base"] = validation_rows["ref"].astype(str).str.upper()
    validation_rows["alt_base"] = validation_rows["alt"].astype(str).str.upper()

    return validation_rows, library_alleles, alleles_lifted, freq_found


validation_rows, library_alleles, alleles_lifted, freq_found = build_validation_rows(
    library_design_df=library_design,
    positions_bed_lift_df=positions_bed_lift,
    freq_table=freq_1kg_table,
)

print(f"library_alleles rows: {len(library_alleles):,}")
print(f"freq_found rows: {len(freq_found):,}")
print(f"validation_rows rows: {len(validation_rows):,}")

In [ ]:
def summarize_hmpra_derived_alleles(validation_rows_df):
    """Summarize whether hMPRA derived allele is REF/ALT in 1KG and return ALT-focused table."""
    analysis_rows = validation_rows_df.copy()

    analysis_rows["hMPRA_derived_allele"] = analysis_rows["library_right"].astype(str).str.upper()
    analysis_rows["ref_base"] = analysis_rows["ref_base"].astype(str).str.upper()
    analysis_rows["alt_base"] = analysis_rows["alt_base"].astype(str).str.upper()
    analysis_rows["AF_num"] = pd.to_numeric(analysis_rows["AF_num"], errors="coerce")

    # Keep only clean SNV rows with available AF for robust comparison
    is_biallelic_snv = (
        analysis_rows["ref_base"].str.len().eq(1)
        & analysis_rows["alt_base"].str.len().eq(1)
        & analysis_rows["ref_base"].str.fullmatch(r"[ACGTN]")
        & analysis_rows["alt_base"].str.fullmatch(r"[ACGTN]")
    )
    has_af = analysis_rows["AF_num"].notna()
    usable = is_biallelic_snv & has_af
    analysis_rows = analysis_rows.loc[usable].copy()

    # Classify each row by where the hMPRA derived allele appears in 1KG
    analysis_rows["hMPRA_role_in_1kg"] = np.select(
        [
            analysis_rows["hMPRA_derived_allele"] == analysis_rows["ref_base"],
            analysis_rows["hMPRA_derived_allele"] == analysis_rows["alt_base"],
        ],
        ["REF", "ALT"],
        default="NEITHER",
    )

    # If hMPRA derived allele is ALT, AF_num is the allele frequency of that derived allele.
    # If it is REF, derived frequency is 1 - AF_num because AF_num refers to ALT frequency.
    analysis_rows["hMPRA_derived_allele_frequency"] = np.where(
        analysis_rows["hMPRA_role_in_1kg"] == "ALT",
        analysis_rows["AF_num"],
        np.where(
            analysis_rows["hMPRA_role_in_1kg"] == "REF",
            1.0 - analysis_rows["AF_num"],
            np.nan,
        ),
    )

    derived_ref_alt_summary = (
        analysis_rows.groupby("hMPRA_role_in_1kg", dropna=False)
        .agg(
            n_rows=("hMPRA_role_in_1kg", "size"),
            mean_hMPRA_derived_allele_frequency=("hMPRA_derived_allele_frequency", "mean"),
            median_hMPRA_derived_allele_frequency=("hMPRA_derived_allele_frequency", "median"),
        )
        .reset_index()
        .sort_values("n_rows", ascending=False)
        .reset_index(drop=True)
    )

    total_compared = len(analysis_rows)
    derived_ref_alt_summary["percent_of_rows"] = np.where(
        total_compared > 0,
        100 * derived_ref_alt_summary["n_rows"] / total_compared,
        np.nan,
    )

    # Ensure explicit REF/ALT/NEITHER rows appear even if count is zero
    for role in ["REF", "ALT", "NEITHER"]:
        if role not in derived_ref_alt_summary["hMPRA_role_in_1kg"].values:
            derived_ref_alt_summary = pd.concat(
                [
                    derived_ref_alt_summary,
                    pd.DataFrame(
                        [{
                            "hMPRA_role_in_1kg": role,
                            "n_rows": 0,
                            "mean_hMPRA_derived_allele_frequency": np.nan,
                            "median_hMPRA_derived_allele_frequency": np.nan,
                            "percent_of_rows": 0.0,
                        }]
                    ),
                ],
                ignore_index=True,
            )

    derived_ref_alt_summary = derived_ref_alt_summary.sort_values(
        by="hMPRA_role_in_1kg",
        key=lambda s: s.map({"REF": 0, "ALT": 1, "NEITHER": 2}),
    ).reset_index(drop=True)

    derived_alt_frequency_table = analysis_rows.loc[
        analysis_rows["hMPRA_role_in_1kg"] == "ALT",
        [
            "hg38_chr", "hg38_start", "hg38_end",
            "variant_token",
            "hMPRA_derived_allele",
            "ref_base", "alt_base",
            "AF_num",
            "hMPRA_derived_allele_frequency",
        ],
    ]
    derived_alt_frequency_table = derived_alt_frequency_table.rename(
        columns={
            "AF_num": "alt_allele_frequency_1kg",
            "hMPRA_derived_allele_frequency": "hMPRA_derived_allele_frequency_1kg",
        }
    ).sort_values(
        ["alt_allele_frequency_1kg", "hg38_chr", "hg38_start"],
        ascending=[False, True, True],
    ).reset_index(drop=True)

    return analysis_rows, derived_ref_alt_summary, derived_alt_frequency_table


analysis_rows, derived_ref_alt_summary, derived_alt_frequency_table = summarize_hmpra_derived_alleles(
    validation_rows_df=validation_rows,
 )

# Console summary
total_compared = len(analysis_rows)
ref_n = int((analysis_rows["hMPRA_role_in_1kg"] == "REF").sum())
alt_n = int((analysis_rows["hMPRA_role_in_1kg"] == "ALT").sum())
neither_n = int((analysis_rows["hMPRA_role_in_1kg"] == "NEITHER").sum())

print("hMPRA derived allele REF/ALT analysis (all usable validation_rows)")
print("=" * 80)
print(f"Usable rows analyzed: {total_compared:,}")
print(f"Identified as REF: {ref_n:,} ({(100 * ref_n / total_compared) if total_compared else np.nan:.2f}%)")
print(f"Identified as ALT: {alt_n:,} ({(100 * alt_n / total_compared) if total_compared else np.nan:.2f}%)")
print(f"Identified as NEITHER: {neither_n:,} ({(100 * neither_n / total_compared) if total_compared else np.nan:.2f}%)")
print()
print("Summary table: derived_ref_alt_summary")
print(derived_ref_alt_summary.to_string(index=False, float_format=lambda x: f"{x:,.6f}"))
print()
print(f"ALT frequency table rows: {len(derived_alt_frequency_table):,}")
print("Table name: derived_alt_frequency_table")
derived_alt_frequency_table

##  HGDP and 1K

In [ ]:
# Apply once on gnomAD v4 table using the single helper function
path_gnomad_3 = "/home/labs/davidgo/Collaboration/humanMPRA/additional/variant_frequency_gnomad_v3_HGDP_and_1KG/positions_bed_lift_gnomad3_frequency.tsv"

freq_1kg_HGDP_table = prepare_frequency_table(path_gnomad_3)

In [ ]:
100-0.0869548139
100-0.0048209034

In [ ]:
zero_af_count = (freq_1kg_HGDP_table["AF_global"] == 0).sum()
total_rows = len(freq_1kg_HGDP_table)
zero_af_percent = (zero_af_count / total_rows) * 100 if total_rows > 0 else np.nan

print(f"Rows with AF_global == 0: {zero_af_count:,}")
print(f"Percent with AF_global == 0: {zero_af_percent:.4f}%")

In [ ]:
sum(freq_1kg_HGDP_table['AF_global']==0)

In [ ]:
duplicated_rows, dup_mask, n_dups, subset_cols = find_duplicated_rows_by_first_cols(freq_1kg_HGDP_table)

In [ ]:
fig, ax, af, af_pos = plot_af_global_histogram(freq_1kg_HGDP_table)
print(f"Plotted {len(af_pos):,} variants with AF_global > 0")
print(f"Excluded {(af <= 0).sum():,} rows with AF_global <= 0")

In [ ]:
validation_rows, library_alleles, alleles_lifted, freq_found = build_validation_rows(
    library_design_df=library_design,
    positions_bed_lift_df=positions_bed_lift,
    freq_table=freq_1kg_HGDP_table,
)

print(f"library_alleles rows: {len(library_alleles):,}")
print(f"freq_found rows: {len(freq_found):,}")
print(f"validation_rows rows: {len(validation_rows):,}")

In [ ]:
analysis_rows, derived_ref_alt_summary, derived_alt_frequency_table = summarize_hmpra_derived_alleles(
    validation_rows_df=validation_rows,
 )

# Console summary
total_compared = len(analysis_rows)
ref_n = int((analysis_rows["hMPRA_role_in_1kg"] == "REF").sum())
alt_n = int((analysis_rows["hMPRA_role_in_1kg"] == "ALT").sum())
neither_n = int((analysis_rows["hMPRA_role_in_1kg"] == "NEITHER").sum())

print("hMPRA derived allele REF/ALT analysis (all usable validation_rows)")
print("=" * 80)
print(f"Usable rows analyzed: {total_compared:,}")
print(f"Identified as REF: {ref_n:,} ({(100 * ref_n / total_compared) if total_compared else np.nan:.2f}%)")
print(f"Identified as ALT: {alt_n:,} ({(100 * alt_n / total_compared) if total_compared else np.nan:.2f}%)")
print(f"Identified as NEITHER: {neither_n:,} ({(100 * neither_n / total_compared) if total_compared else np.nan:.2f}%)")
print()
print("Summary table: derived_ref_alt_summary")
print(derived_ref_alt_summary.to_string(index=False, float_format=lambda x: f"{x:,.6f}"))
print()
print(f"ALT frequency table rows: {len(derived_alt_frequency_table):,}")
print("Table name: derived_alt_frequency_table")
derived_alt_frequency_table